# Stage 3 — feature assembler + rolling-origin split

This notebook is the Stage 3 demo surface. It assembles the first end-to-end feature table — the hourly join of NESO demand with a population-weighted national weather aggregate — and enumerates rolling-origin train/test folds against it.

### Why a feature assembler?

Stage 2 left the demand/weather join inline in the notebook "so Stage 3 has a home to move it into". That home is `bristol_ml.features.assembler`: one canonical `build()` function, one declared `OUTPUT_SCHEMA`, one atomic parquet on disk. Every downstream stage (baselines, tree models, LLM prompts) reads the same file; no notebook reimplements the join.

### Why rolling-origin?

Random train/test splits leak future into past for a time series. Rolling-origin (Tashman 2000; "walk-forward" in ML practitioner usage) preserves time order: the origin advances stepwise, every fold trains on a prefix of history and tests on an immediately-following horizon. It is also the natural shape for reporting mean-and-spread across folds per DESIGN §5.3.

### What this notebook does

1. Resolves the project config via Hydra.
2. Calls `assembler.assemble(cfg, cache="auto")` — this warm-runs the ingestion layer and writes `data/features/weather_only.parquet`. Re-runs are cache hits.
3. Loads the feature table back through `assembler.load(path)` — schema-validated read.
4. Enumerates rolling-origin folds against it using the resolved `evaluation.rolling_origin` config.
5. Plots the demand series with fold boundaries overlaid, in Europe/London local time for legibility.

Acceptance criterion 5: this runs end-to-end in under 120 seconds on a laptop with warm caches (D7 — see `docs/plans/completed/03-feature-assembler.md` §1).


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # cache_dir values resolve against cwd

import matplotlib.dates as mdates  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

from bristol_ml import load_config  # noqa: E402
from bristol_ml.evaluation.splitter import rolling_origin_split_from_config  # noqa: E402
from bristol_ml.features import assembler  # noqa: E402

# The resolved `conf/evaluation/rolling_origin.yaml` default picks
# `min_train_periods=8760` (one full year of hourly data) which assumes
# a multi-year training window. The pre-populated Stage 2 weather cache
# covers one calendar year, so the assembled feature table is ~8760 rows.
# A Hydra override narrows the first-fold training window to 30 days
# (720 h) so every fold in this notebook is feasible and the demo is
# fast. Production runs over a multi-year cache would drop this line.
cfg = load_config(
    config_path=REPO_ROOT / "conf",
    overrides=["evaluation.rolling_origin.min_train_periods=720"],
)
assert cfg.features.weather_only is not None, "features.weather_only not resolved"
assert cfg.evaluation.rolling_origin is not None, "evaluation.rolling_origin not resolved"

# Build (or reuse) the feature-set parquet. `cache="auto"` will populate both
# ingestion caches on first run and become a pair of cache hits on re-runs.
feature_path = assembler.assemble(cfg, cache="auto")
print(f"Feature table at: {feature_path}")

## The assembled feature table

`assembler.load` round-trips the parquet through pyarrow with the declared schema asserted column-by-column. An extra or missing column here is a hard error — downstream models may select columns positionally.


In [ ]:
features = assembler.load(feature_path)
print(f"Feature table: {len(features):,} rows, {len(features.columns)} columns")
print(f"Time range: {features['timestamp_utc'].min()}  →  {features['timestamp_utc'].max()}")
print("\nDtypes:")
print(features.dtypes)
print("\nHead:")
features.head()

## Rolling-origin folds

The resolved `evaluation.rolling_origin` config (with the in-notebook override) gives us:

- `min_train_periods=720` (30 days — overridden above; the ship default is 8760 for a multi-year window).
- `test_len=24` (one day-ahead horizon per fold).
- `step=24` (non-overlapping daily folds).
- `gap=0` (no embargo — day-ahead forecasters see yesterday's actuals at gate closure).
- `fixed_window=False` (expanding window — every fold sees all history to date).

`rolling_origin_split` yields `(train_idx, test_idx)` pairs of `numpy.int64` arrays. Downstream code slices its own DataFrame via `df.iloc[idx]` — the splitter is data-structure-agnostic (AC-4).


In [ ]:
splitter_cfg = cfg.evaluation.rolling_origin
folds = list(rolling_origin_split_from_config(len(features), splitter_cfg))
print(f"Config: {splitter_cfg.model_dump()}")
print(f"Fold count: {len(folds):,}")
print(
    f"First fold:  train[0..2]={folds[0][0][:3].tolist()}  "
    f"test[0..2]={folds[0][1][:3].tolist()}  "
    f"(train_len={len(folds[0][0]):,}, test_len={len(folds[0][1])})"
)
print(
    f"Last fold:   train[-3:]={folds[-1][0][-3:].tolist()}  "
    f"test[-3:]={folds[-1][1][-3:].tolist()}  "
    f"(train_len={len(folds[-1][0]):,}, test_len={len(folds[-1][1])})"
)

## Fold boundaries on the demand series

We plot `nd_mw` over time and overlay the first test fold's boundary (where training ends, testing begins) plus the last test fold's boundary. The expanding-window shape is visible: fold 1 trains on the leftmost 30 days; the last fold trains on almost everything to the left of its own test window.

Localisation: `timestamp_utc` stays canonical for arithmetic; the **display** axis uses `Europe/London` so a meetup audience sees wall-clock time. `timestamp_local` on disk is *never* used for joins — Gotcha 2 in the Stage 3 exploration notes.


In [ ]:
display_ts = features["timestamp_utc"].dt.tz_convert("Europe/London")

first_train, first_test = folds[0]
last_train, last_test = folds[-1]
first_boundary = display_ts.iloc[first_test[0]]
last_boundary = display_ts.iloc[last_test[0]]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(display_ts, features["nd_mw"], linewidth=0.4, color="C0", alpha=0.75)
ax.axvspan(
    display_ts.iloc[first_train[0]],
    first_boundary,
    color="C2",
    alpha=0.08,
    label=f"Fold 1 train ({len(first_train):,} h)",
)
ax.axvline(first_boundary, color="C2", linewidth=1.2, linestyle="--", label="Fold 1 test origin")
ax.axvline(
    last_boundary,
    color="C3",
    linewidth=1.2,
    linestyle="--",
    label=f"Fold {len(folds)} test origin",
)
ax.set_xlabel("Local time (Europe/London)")
ax.set_ylabel("National Demand (MW)")
ax.set_title("Rolling-origin folds overlaid on GB National Demand")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Fold anatomy in a single week

Zooming in on the first few test folds shows the day-ahead shape: each red bar is one 24-hour test horizon; the space between bars is the next day's training increment (`step=24`). `step==test_len` means test windows are non-overlapping — the standard for day-ahead evaluation.


In [ ]:
n_preview_folds = 7
preview_folds = folds[:n_preview_folds]
preview_start = display_ts.iloc[preview_folds[0][1][0] - 12]
preview_end = display_ts.iloc[preview_folds[-1][1][-1] + 12]
mask = (display_ts >= preview_start) & (display_ts <= preview_end)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(display_ts[mask], features.loc[mask, "nd_mw"], color="C0", linewidth=1.0)
for i, (_train, test) in enumerate(preview_folds):
    start = display_ts.iloc[test[0]]
    end = display_ts.iloc[test[-1]]
    ax.axvspan(start, end, color="C3", alpha=0.18, label="Test fold" if i == 0 else None)

ax.set_xlabel("Local time (Europe/London)")
ax.set_ylabel("National Demand (MW)")
ax.set_title(f"First {n_preview_folds} day-ahead test folds")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## What this sets up

From here on, every Stage 4+ model reads `data/features/weather_only.parquet` and enumerates folds with `rolling_origin_split_from_config`. The first baseline (Stage 4) is a linear regression on the five weather columns against `nd_mw`, evaluated fold-by-fold with MAE / MAPE / RMSE / WAPE; the Stage 5 `weather_calendar` set adds calendar features and the comparison against this `weather_only` baseline becomes a config swap, not a code change.

### Things to try in a live demo

- Override `features.weather_only.demand_aggregation=max` at the CLI (or re-import and re-run) to switch to a peak-demand framing per decision D1. Shapes change; the rolling-origin fold indices do not.
- Switch to a sliding training window by overriding `evaluation.rolling_origin.fixed_window=true`. Fold count stays the same; `len(train)` becomes constant.
- Tighten `evaluation.rolling_origin.min_train_periods` further (say to 168, one week) and watch the fold count climb — useful for a rapid-iteration debugging loop, not for reporting.
